# GROBID PDF Parsing + Pinecone Embedding

In [1]:
from langchain_community.document_loaders.parsers import GrobidParser
from langchain_community.document_loaders.generic import GenericLoader

/Users/a215482/Desktop/personal projects/rag-citation-langchain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
try:
    parser = GrobidParser(segment_sentences=False)
    print("Parser created")
    print(parser)
except Exception as e:
    print(f"Error: {e}")


Parser created


In [3]:

loader = GenericLoader.from_filesystem(
    ".",
    glob="hope.pdf",
    suffixes=[".pdf"],
    parser=GrobidParser(segment_sentences=False)
)

In [4]:
try :
    loader.load()
except Exception as e:
    print(f"Error: {e}")


In [5]:
docs = loader.load()

In [6]:
docs[0].metadata

{'text': "We follow Baron's (2008) seminal work in defining the relevant terms here.Affect represents the emotions, moods, and passions of the entrepreneurit is the realized subjective feelings of (dis)pleasure attached to an object (e.g., experience; physical thing), where such feelings that can differ in length, intensity, antecedent, and so on.The entrepreneurial process consists of creatively identifying an opportunity and acting to exploit it, all under uncertainty (where the opportunity can be identified ex post as a market imperfection believed ex ante to indicate that unrealizable value existed from which some party can profit).That process is often characterized by multiple stages, non-routineness, and imperfection.Because such characterizations are associated with higher levels of emotion (e.g., where the personal stakes and pressures are greater -Cardon et al., 2012), the entrepreneurial process requires attention be paid to affect.Baron (2008) highlights the many potential 

In [7]:
from langchain_text_splitters import CharacterTextSplitter
from models import Chunk

In [8]:
chunks: list[Chunk] = []
for doc in docs:
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base", chunk_size=1500, chunk_overlap=300
    )
    texts = text_splitter.split_text(doc.page_content)
    for i, text in enumerate(texts):
        chunk = Chunk(
            text=text,
            metadata=doc.metadata
        )
        chunks.append(chunk)

chunks


[Chunk(text="We follow Baron's (2008) seminal work in defining the relevant terms here.Affect represents the emotions, moods, and passions of the entrepreneurit is the realized subjective feelings of (dis)pleasure attached to an object (e.g., experience; physical thing), where such feelings that can differ in length, intensity, antecedent, and so on.The entrepreneurial process consists of creatively identifying an opportunity and acting to exploit it, all under uncertainty (where the opportunity can be identified ex post as a market imperfection believed ex ante to indicate that unrealizable value existed from which some party can profit).That process is often characterized by multiple stages, non-routineness, and imperfection.Because such characterizations are associated with higher levels of emotion (e.g., where the personal stakes and pressures are greater -Cardon et al., 2012), the entrepreneurial process requires attention be paid to affect.Baron (2008) highlights the many potenti

In [9]:
import voyageai
from dotenv import load_dotenv
from typing import List

load_dotenv()

def get_voyage_client():

    return voyageai.Client()

In [10]:
vo = get_voyage_client()

In [11]:
from models import Embedding 

embeddingsList: List[Embedding] = []
inputs: List[str] = []
for chunk in chunks:
    inputs.append(chunk.text)


# Get embeddings from Voyage AI
raw_embeddings = vo.embed(
    texts=inputs, model="voyage-4-lite", input_type="document"
)

for i, raw_embedding in enumerate(raw_embeddings.embeddings):
    embedding = Embedding(
        chunk=chunks[i],
        embedding=raw_embedding
    )
    embeddingsList.append(embedding)


In [17]:
from langchain_voyageai import VoyageAIEmbeddings
import os


embeddings = VoyageAIEmbeddings(
    voyage_api_key=os.getenv("VOYAGE_API_KEY"), model="voyage-4-lite"
)


In [18]:
from pinecone import Pinecone
from dotenv import load_dotenv



load_dotenv()

def get_pinecone_client():
    return Pinecone()

pc = get_pinecone_client()

In [29]:
from pinecone import ServerlessSpec

index_name = "langchain-test-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [30]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)


In [31]:
from uuid import uuid4

from langchain_core.documents import Document

documents: list[Document] = []

for chunk in chunks:
    documents.append(Document(page_content=chunk.text, metadata=chunk.metadata))
uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)


['3906babb-9f42-47b5-a38b-9dddcae61142',
 '25a8c2d0-e1d0-4732-8c61-fb7607f92003',
 'c640bac7-94ec-4eac-9ee0-88c1710316ac',
 'ad40d64d-5138-4490-b7a0-5d8f859b5fb7',
 '760f573a-46bb-45b6-9b00-f78beea6fb0c',
 '6c992522-76af-438a-a765-43b4eeb9e1cb',
 '32f3513d-757d-4d2e-b4e1-cd3a5afa416c',
 'f9f60798-3295-4339-ae64-46960280ee6b',
 '4118ffea-922e-4e26-b607-4f2542a14887',
 'bfafb594-03a9-41f1-a06c-fe0ebd5c7511',
 'c6d7cdfd-5648-4293-8433-5847fb222f89',
 '1756660b-b1db-4005-8ae4-309a76397daf',
 '1ae2223d-9a01-4f74-81ff-f7c8e7de7bdd',
 'bf2bec81-9742-4a1c-a841-81feb48a6515']

In [37]:
retriever = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.5}
    )

In [40]:
retriever.invoke("Hope is the emotion that drives the teams, and makes them decide if they will quit or not quit")

AttributeError: module 'langchain' has no attribute 'debug'